In [4]:
import os
import glob
import json
import numpy as np
import onnx
import onnxruntime as ort
from PIL import Image
from tqdm import tqdm
from tokenizers import Tokenizer

In [5]:
# for some reason initializing this speeds up the token decoder
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("./weights/model_best_st_224x224/", trust_remote_code=True) 

In [6]:
class FastTokenizer:
    def __init__(self, tokenizer_blob):
        meta = tokenizer_blob.pop("meta")
        self.tokenizer = Tokenizer.from_str(json.dumps(tokenizer_blob))
        self.special_ids = set(meta["special_ids"])
        self.eos_token_id = meta["eos_token_id"]

    def decode(self, ids):
        ids = np.asarray(ids)

        # flatten if needed
        if ids.ndim > 1:
            ids = ids.reshape(-1)

        # stop at EOS
        eos_pos = np.where(ids == self.eos_token_id)[0]
        if eos_pos.size > 0:
            ids = ids[:eos_pos[0]]

        # remove special tokens
        if self.special_ids:
            mask = ~np.isin(ids, list(self.special_ids))
            ids = ids[mask]

        return self.tokenizer.decode(ids.tolist())


class Florence2ONNX:
    def __init__(
        self,
        encoder_path,
        decoder_path,
        device="cuda",
        max_new_tokens=15,
    ):
        self.max_new_tokens = max_new_tokens

        # tokens
        self.bos_token_id = 0
        self.eos_token_id = 2

        # normalization
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        self.rescale = 1.0 / 255.0

        # load tokenizer from ONNX metadata (do once)
        for prop in onnx.load(decoder_path).metadata_props:
            if prop.key == "tokenizer_json":
                self.tokenizer = FastTokenizer(json.loads(prop.value))
                break

        # ONNX runtime setup
        providers = (
            [("CUDAExecutionProvider", {}), "CPUExecutionProvider"]
            if device == "cuda"
            else ["CPUExecutionProvider"]
        )

        sess_opts = ort.SessionOptions()
        sess_opts.enable_mem_pattern = False

        self.encoder_sess = ort.InferenceSession(
            encoder_path, sess_options=sess_opts, providers=providers
        )

        for inp in self.encoder_sess.get_inputs():
            if inp.name == "pixel_values":
                self.image_size = inp.shape[-1]

        self.decoder_sess = ort.InferenceSession(
            decoder_path, sess_options=sess_opts, providers=providers
        )
        
        # reusable decoder input dict
        self.decoder_inputs = {
            "encoder_hidden_states": None,
            "decoder_input_ids": None,
        }

    def _preprocess_image(self, image: Image.Image):
        if image.mode != "RGB":
            image = image.convert("RGB")

        image = image.resize(
            (self.image_size, self.image_size),
            Image.Resampling.BICUBIC,
        )

        img = np.asarray(image, dtype=np.float32)

        img *= self.rescale
        img -= self.mean
        img /= self.std

        img = img.transpose(2, 0, 1)
        return np.ascontiguousarray(img[None, ...])

    def infer_batch(self, images):
        batch_size = len(images)

        pixel_values = np.concatenate(
            [self._preprocess_image(img) for img in images],
            axis=0
        ).astype(np.float16)  # (N, C, H, W)

        prompt = [0, 2264, 16, 5, 2788, 11, 5, 2274, 116, 2]
        input_ids = np.tile(
            np.array(prompt, dtype=np.int64)[None, :],
            (batch_size, 1)
        )

        encoder_hidden = self.encoder_sess.run(
            ["encoder_hidden_states"],
            {
                "input_ids": input_ids,
                "pixel_values": pixel_values
            }
        )[0].astype(np.float16)

        generated_ids = np.full(
            (batch_size, self.max_new_tokens + 1),
            fill_value=self.eos_token_id,
            dtype=np.int64
        )

        generated_ids[:, 0] = self.bos_token_id
        cur_len = 1

        finished = np.zeros(batch_size, dtype=bool)

        for _ in range(self.max_new_tokens):
            logits = self.decoder_sess.run(
                ["logits"],
                {
                    "decoder_input_ids": generated_ids[:, :cur_len],
                    "encoder_hidden_states": encoder_hidden,
                },
            )[0]  # (B, T, V)

            next_tokens = np.argmax(logits[:, -1, :], axis=-1)  # (B,)

            generated_ids[:, cur_len] = next_tokens
            cur_len += 1

            # track EOS per sample
            finished |= (next_tokens == self.eos_token_id)

            if finished.all():
                break

        outputs = []
        for i in range(batch_size):
            outputs.append(self.tokenizer.decode(generated_ids[i]))

        return outputs

In [7]:
onnx_model = Florence2ONNX(
    encoder_path="florence2_encoder_fp16.onnx",
    decoder_path="florence2_decoder_fp16.onnx",
    device="cuda"
)

/usr/local/lib/python3.10/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [ ]:
import glob

images_path = glob.glob("./dataset/lp_crops_st_balanced/val/*.jpg")

for image_path in images_path[:5]:
    with open(image_path.replace(".jpg", ".txt")) as f:
        gt = f.read().strip()

    image = Image.open(image_path)
    pred = onnx_model.infer_batch([image])[0]

    display(image)
    print("gt:", gt)
    print("pr:", pred, "\n")
    break

In [ ]:
BATCH_SIZE = 32

def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row

    return prev_row[-1]


total, d0, d1, d2 = 0, 0, 0, 0

images_path = glob.glob("./dataset/lp_crops_st/*.jpg")

batch_images = []
batch_gts = []

for image_path in tqdm(images_path):
    # load GT
    gt_path = image_path.replace(".jpg", ".txt")
    
    if not (os.path.exists(gt_path)):
        continue

    with open(gt_path) as f:
        gt = f.read().strip()
        if not gt:
            continue

    # load image
    image = Image.open(image_path)

    batch_images.append(image)
    batch_gts.append(gt)

    # run batch
    if len(batch_images) == BATCH_SIZE:
        preds = onnx_model.infer_batch(batch_images)

        for gt, pred in zip(batch_gts, preds):
            dist = levenshtein_distance(
                gt.lower(),
                pred.replace(" ", "").lower()
            )

            total += 1
            if dist == 0:
                d0 += 1
            if dist <= 1:
                d1 += 1
            if dist <= 2:
                d2 += 1

        batch_images.clear()
        batch_gts.clear()


# handle leftover batch
if batch_images:
    preds = onnx_model.infer_batch(batch_images)

    for gt, pred in zip(batch_gts, preds):
        if not gt:
            continue

        dist = levenshtein_distance(
            gt.lower(),
            pred.replace(" ", "").lower()
        )

        total += 1
        if dist == 0:
            d0 += 1
        if dist <= 1:
            d1 += 1
        if dist <= 2:
            d2 += 1


# metrics
acc_exact = d0 / total
acc_1 = d1 / total
acc_2 = d2 / total

print(f"Exact accuracy (distance=0): {acc_exact:.4f}")
print(f"Accuracy (distance<=1):     {acc_1:.4f}")
print(f"Accuracy (distance<=2):     {acc_2:.4f}")